In [3]:
import boto3
import json
import time
import zipfile
from io import BytesIO
import uuid
import pprint
import logging
print(boto3.__version__)

1.43.83


In [2]:
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

In [6]:
sts_client = boto3.client('sts')
iam_client = boto3.client('iam')
lambda_client = boto3.client('lambda')
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime')

In [7]:
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()["Account"]
region, account_id

('us-east-1', '938176218293')

In [6]:
inference_profile = "us.amazon.nova-micro-v1:0" 
foundation_model = inference_profile[3:]
foundation_model

'amazon.nova-micro-v1:0'

In [7]:
suffix = f"{region}-{account_id}"
agent_name = "hr-assistant-function-def"
agent_bedrock_allow_policy_name = f"{agent_name}-ba-{suffix}"
agent_role_name = f'AmazonBedrockExecutionRoleForAgents_{agent_name}'
agent_description = "Agent for providing HR assistance to manage vacation time"
agent_instruction = "You are an HR agent, helping employees understand HR policies and manage vacation time"
agent_action_group_name = "VacationsActionGroup"
agent_action_group_description = "Actions for getting the number of available vacations days for an employee and confirm new time off"
agent_alias_name = f"{agent_name}-alias"
lambda_function_role = f'{agent_name}-lambda-role-{suffix}'
lambda_function_name = f'{agent_name}-{suffix}'

In [12]:
# creating employee database to be used by lambda function
import sqlite3
import random
from datetime import date, timedelta

# Connect to the SQLite database (creates a new one if it doesn't exist)
conn = sqlite3.connect('employee_database.db')
c = conn.cursor()

# Create the employees table
c.execute('''CREATE TABLE IF NOT EXISTS employees
                (employee_id INTEGER PRIMARY KEY AUTOINCREMENT, employee_name TEXT, employee_job_title TEXT, employee_start_date TEXT, employee_employment_status TEXT)''')

# Create the vacations table
c.execute('''CREATE TABLE IF NOT EXISTS vacations
                (employee_id INTEGER, year INTEGER, employee_total_vacation_days INTEGER, employee_vacation_days_taken INTEGER, employee_vacation_days_available INTEGER, FOREIGN KEY(employee_id) REFERENCES employees(employee_id))''')

# Create the planned_vacations table
c.execute('''CREATE TABLE IF NOT EXISTS planned_vacations
                (employee_id INTEGER, vacation_start_date TEXT, vacation_end_date TEXT, vacation_days_taken INTEGER, FOREIGN KEY(employee_id) REFERENCES employees(employee_id))''')

# Generate some random data for 10 employees
employee_names = ['John Doe', 'Jane Smith', 'Bob Johnson', 'Alice Williams', 'Tom Brown', 'Emily Davis', 'Michael Wilson', 'Sarah Taylor', 'David Anderson', 'Jessica Thompson']
job_titles = ['Manager', 'Developer', 'Designer', 'Analyst', 'Accountant', 'Sales Representative']
employment_statuses = ['Active', 'Inactive']

for i in range(10):
    name = employee_names[i]
    job_title = random.choice(job_titles)
    start_date = date(2015 + random.randint(0, 7), random.randint(1, 12), random.randint(1, 28)).strftime('%Y−%m−%d')
    employment_status = random.choice(employment_statuses)
    c.execute("INSERT INTO employees (employee_name, employee_job_title, employee_start_date, employee_employment_status) VALUES (?, ?, ?, ?)", (name, job_title, start_date, employment_status))
    employee_id = c.lastrowid

    # Generate vacation data for the current employee
    for year in range(date.today().year, date.today().year - 3, -1):
        total_vacation_days = random.randint(10, 30)
        days_taken = random.randint(0, total_vacation_days)
        days_available = total_vacation_days - days_taken
        c.execute("INSERT INTO vacations (employee_id, year, employee_total_vacation_days, employee_vacation_days_taken, employee_vacation_days_available) VALUES (?, ?, ?, ?, ?)", (employee_id, year, total_vacation_days, days_taken, days_available))

        # Generate some planned vacations for the current employee and year
        num_planned_vacations = random.randint(0, 3)
        for _ in range(num_planned_vacations):
            start_date = date(year, random.randint(1, 12), random.randint(1, 28)).strftime('%Y−%m−%d')
            end_date = (date(int(start_date[:4]), int(start_date[5:7]), int(start_date[8:])) + timedelta(days=random.randint(1, 14))).strftime('%Y−%m−%d')
            days_taken = (date(int(end_date[:4]), int(end_date[5:7]), int(end_date[8:])) - date(int(start_date[:4]), int(start_date[5:7]), int(start_date[8:])))
            c.execute("INSERT INTO planned_vacations (employee_id, vacation_start_date, vacation_end_date, vacation_days_taken) VALUES (?, ?, ?, ?)", (employee_id, start_date, end_date, days_taken.days))

# Commit the changes and close the connection
conn.commit()
conn.close()

In [47]:
%%writefile lambda_function.py

import os
import json
import shutil
import sqlite3


def get_available_vacations_days(employee_id):

    db_path = "/tmp/employee_database.db"

    # Copy database from Lambda deployment package to /tmp
    if not os.path.exists(db_path):
        shutil.copy(
            os.path.join(
                os.path.dirname(__file__),
                "employee_database.db"
            ),
            db_path
        )

    conn = sqlite3.connect(db_path)

    try:
        cursor = conn.cursor()

        if not employee_id:
            raise ValueError("No employee id provided")

        cursor.execute("""
            SELECT employee_vacation_days_available
            FROM vacations
            WHERE employee_id = ?
            ORDER BY year DESC
            LIMIT 1
        """, (employee_id,))

        result = cursor.fetchone()

        if result:
            return result[0]

        return f"No vacation data found for employee_id {employee_id}"

    finally:
        conn.close()


def lambda_handler(event, context):

    print("Gateway event:")
    print(json.dumps(event))

    # ----------------------------------------
    # AgentCore Gateway event
    # ----------------------------------------
    if "tool_name" in event:

        tool_name = event["tool_name"]

        # Gateway may prefix the tool name with the target name.
        # We only need the actual tool name.
        tool_name = tool_name.split("__")[-1]

        if tool_name == "get_available_vacation_days":

            arguments = event.get("arguments", {})

            employee_id = arguments.get("employee_id")

            available_days = get_available_vacations_days(
                employee_id
            )

            return {
                "available_vacation_days": available_days
            }

        raise ValueError(
            f"Unknown tool: {tool_name}"
        )

    # ----------------------------------------
    # Direct Lambda invocation
    # ----------------------------------------
    employee_id = event.get("employee_id")

    available_days = get_available_vacations_days(
        employee_id
    )

    return {
        "statusCode": 200,
        "body": json.dumps({
            "employee_id": employee_id,
            "available_vacation_days": available_days
        })
    }

Overwriting lambda_function.py


In [14]:

try:
    assume_role_policy_document = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": "lambda.amazonaws.com"
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }

    assume_role_policy_document_json = json.dumps(assume_role_policy_document)

    lambda_iam_role = iam_client.create_role(
        RoleName=lambda_function_role,
        AssumeRolePolicyDocument=assume_role_policy_document_json
    )

    # Pause to make sure role is created
    time.sleep(10)
except:
    lambda_iam_role = iam_client.get_role(RoleName=lambda_function_role)

iam_client.attach_role_policy(
    RoleName=lambda_function_role,
    PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
)

{'ResponseMetadata': {'RequestId': '0c58b749-75f5-4f3e-89e0-9aa0dcc7c924',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 31 Aug 2026 08:54:14 GMT',
   'x-amzn-requestid': '0c58b749-75f5-4f3e-89e0-9aa0dcc7c924',
   'content-type': 'text/xml',
   'content-length': '212'},
  'RetryAttempts': 0}}

In [33]:

# Package up the lambda function code
# s = BytesIO()
# z = zipfile.ZipFile(s, 'w')
# z.write("lambda_function.py")
# z.write("employee_database.db")
# z.close()
# zip_content = s.getvalue()

# Create Lambda Function
# lambda_function = lambda_client.create_function(
#     FunctionName=lambda_function_name,
#     Runtime='python3.12',
#     Timeout=180,
#     Role=lambda_iam_role['Role']['Arn'],
#     Code={'ZipFile': zip_content},
#     Handler='lambda_function.lambda_handler'
# )

In [49]:
# update lambda dunction
from io import BytesIO
import zipfile

s = BytesIO()

with zipfile.ZipFile(s, "w") as z:
    z.write("lambda_function.py")
    z.write("employee_database.db")

zip_content = s.getvalue()

lambda_update = lambda_client.update_function_code(
    FunctionName=lambda_function_name,
    ZipFile=zip_content
)

print("Lambda updated successfully")

Lambda updated successfully


In [50]:
response = lambda_client.invoke(
    FunctionName=lambda_function_name,
    InvocationType="RequestResponse",
    Payload=json.dumps({
        "employee_id": 1
    })
)

payload = response["Payload"].read().decode("utf-8")

print(payload)

{"statusCode": 200, "body": "{\"employee_id\": 1, \"available_vacation_days\": 17}"}


In [37]:
payload = response["Payload"].read().decode("utf-8")
print(payload)

{"statusCode": 200, "body": "{\"employee_id\": 1, \"available_vacation_days\": 17}"}


In [16]:
bedrock_agent_bedrock_allow_policy_statement = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AmazonBedrockAgentBedrockFoundationModelPolicy",
            "Effect": "Allow",
            "Action": "bedrock:InvokeModel",
            "Resource": [
                f"arn:aws:bedrock:*::foundation-model/{foundation_model}",
                f"arn:aws:bedrock:*:*:inference-profile/{inference_profile}"
            ]
        },
        {
            "Sid": "AmazonBedrockAgentBedrockGetInferenceProfile",
            "Effect": "Allow",
            "Action":  [
                "bedrock:GetInferenceProfile",
                "bedrock:ListInferenceProfiles",
                "bedrock:UseInferenceProfile"
            ],
            "Resource": [
                f"arn:aws:bedrock:*:*:inference-profile/{inference_profile}"
            ]
        }
    ]
}

bedrock_policy_json = json.dumps(bedrock_agent_bedrock_allow_policy_statement)

agent_bedrock_policy = iam_client.create_policy(
    PolicyName=agent_bedrock_allow_policy_name,
    PolicyDocument=bedrock_policy_json
)

In [17]:

# Create IAM Role for the agent and attach IAM policies
assume_role_policy_document = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {
            "Service": "bedrock.amazonaws.com"
        },
        "Action": "sts:AssumeRole"
    }]
}

assume_role_policy_document_json = json.dumps(assume_role_policy_document)
agent_role = iam_client.create_role(
    RoleName=agent_role_name,
    AssumeRolePolicyDocument=assume_role_policy_document_json
)

# Pause to make sure role is created
time.sleep(10)

iam_client.attach_role_policy(
    RoleName=agent_role_name,
    PolicyArn=agent_bedrock_policy['Policy']['Arn']
)

{'ResponseMetadata': {'RequestId': 'd5a6c924-b6a7-4088-a4e1-b4434c24c47a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 31 Aug 2026 09:04:05 GMT',
   'x-amzn-requestid': 'd5a6c924-b6a7-4088-a4e1-b4434c24c47a',
   'content-type': 'text/xml',
   'content-length': '212'},
  'RetryAttempts': 0}}

In [29]:
# response = bedrock_agent_client.create_agent(
#     agentName=agent_name,
#     agentResourceRoleArn=agent_role['Role']['Arn'],
#     description=agent_description,
#     idleSessionTTLInSeconds=1800,
#     foundationModel=inference_profile,
#     instruction=agent_instruction,
# )
# agent_id = response['agent']['agentId']
# agent_id, response

In [38]:
%pip install -U bedrock-agentcore

  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/15.7 MB ? eta -:--:--
   ------ --------------------------------- 2.4/15.7 MB 12.3 MB/s eta 0:00:02
   ----------- ---------------------------- 4.5/15.7 MB 12.2 MB/s eta 0:00:01
   -------------- ------------------------- 5.5/15.7 MB 9.1 MB/s eta 0:00:02
   ---------------- ----------------------- 6.3/15.7 MB 7.6 MB/s eta 0:00:02
   ----------------- ---------------------- 6.8/15.7 MB 6.7 MB/s eta 0:00:02
   ------------------ --------------------- 7.3/15.7 MB 5.8 MB/s eta 0:00:02
   -------------------- ------------------- 7.9/15.7 MB 5.2 MB/s eta 0:00:02
   -------------------- ------------------- 8.1/15.7 MB 5.0 MB/s eta 0:00:02
   ---------------------- ----------------- 8.7/15.7 MB 4.7 MB/s eta 0:00:02
   ----------------------- ---------------- 9.2/15.7 MB 4.5 MB/s eta 0:00:02
   ------------------------ --------------- 9.7/15.7 MB 4.3 MB/s eta 0:00:02
   -------


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


##### Strands Agents + AgentCore Runtime + your existing Lambda

In [39]:
%pip install -U strands-agents

  Using cached docstring_parser-0.18.0-py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/728.3 kB ? eta -:--:--
   ---------------------------------------- 728.3/728.3 kB 4.2 MB/s eta 0:00:00
Using cached docstring_parser-0.18.0-py3-none-any.whl (22 kB)
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ------------------- -------------------- 3.4/6.9 MB 18.4 MB/s eta 0:00:01
   ------------------------- -------------- 4.5/6.9 MB 10.3 MB/s eta 0:00:01
   ------------------------------ --------- 5.2/6.9 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------  6.8/6.9 MB 8.1 MB/s eta 0:00:01
   ---------------------------------------- 6.9/6.9 MB 7.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ------------- -------------------------- 1.3/3.8 MB 6.7 MB/s eta 0:00:01
   ------------------------ --------------- 2.4/3.8 MB 6.1 MB/s eta 0:00:01
   -------------------------------- -----


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
from strands import Agent, tool

print("Strands imported successfully")

Strands imported successfully


###### Create an AgentCore Gateway IAM role
bedrock-agentcore.amazonaws.com

In [10]:
gateway_role_name = f"AgentCoreGatewayRole-{region}"

gateway_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock-agentcore.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

gateway_role = iam_client.create_role(
    RoleName=gateway_role_name,
    AssumeRolePolicyDocument=json.dumps(gateway_trust_policy),
    Description="IAM role for Amazon Bedrock AgentCore Gateway"
)

gateway_role_arn = gateway_role["Role"]["Arn"]

gateway_role_arn

EntityAlreadyExistsException: An error occurred (EntityAlreadyExists) when calling the CreateRole operation: Role with name AgentCoreGatewayRole-us-east-1 already exists.

##### Give Gateway permission to invoke your Lambda

In [42]:
lambda_arn = lambda_function["FunctionArn"]

gateway_lambda_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "lambda:InvokeFunction"
            ],
            "Resource": lambda_arn
        }
    ]
}

gateway_lambda_policy_response = iam_client.create_policy(
    PolicyName=f"AgentCoreGatewayLambdaInvoke-{region}",
    PolicyDocument=json.dumps(gateway_lambda_policy),
    Description="Allow AgentCore Gateway to invoke the HR Lambda"
)

gateway_lambda_policy_arn = (
    gateway_lambda_policy_response["Policy"]["Arn"]
)

gateway_lambda_policy_arn

'arn:aws:iam::938176218293:policy/AgentCoreGatewayLambdaInvoke-us-east-1'

In [43]:
import time

time.sleep(10)

print("IAM role ready")

IAM role ready


#### Create the AgentCore Gateway

In [8]:
agentcore_control_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region
)

In [9]:
gateway_response = agentcore_control_client.create_gateway(
    name="hr-assistant-gateway",
    description="Gateway for HR vacation management tools",
    roleArn=gateway_role_arn,
    protocolType="MCP",
    authorizerType="NONE"
)

gateway_response

NameError: name 'gateway_role_arn' is not defined

In [12]:
gateway_id = gateway_response["gatewayId"]
gateway_arn = gateway_response["gatewayArn"]
gateway_url = gateway_response["gatewayUrl"]

print("Gateway ID:", gateway_id)
print("Gateway ARN:", gateway_arn)
print("Gateway URL:", gateway_url)

NameError: name 'gateway_response' is not defined

#### Define the Gateway tool schema

In [60]:
tool_schema = [
    {
        "name": "get_available_vacations_days",
        "description": "Gets the number of available vacation days for an employee.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "employee_id": {
                    "type": "integer",
                    "description": "The employee ID."
                }
            },
            "required": ["employee_id"]
        }
    }
]

print(type(tool_schema))

<class 'list'>


In [64]:
iam_client.attach_role_policy(
    RoleName=gateway_role_name,
    PolicyArn=gateway_lambda_policy_arn
)

print("Policy attached successfully")

Policy attached successfully


In [65]:
iam_client.list_attached_role_policies(
    RoleName=gateway_role_name
)

{'AttachedPolicies': [{'PolicyName': 'AgentCoreGatewayLambdaInvoke-us-east-1',
   'PolicyArn': 'arn:aws:iam::938176218293:policy/AgentCoreGatewayLambdaInvoke-us-east-1'}],
 'IsTruncated': False,
 'ResponseMetadata': {'RequestId': '9b801604-de4d-4fe4-92e6-f4bf42802cb5',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 31 Aug 2026 10:58:46 GMT',
   'x-amzn-requestid': '9b801604-de4d-4fe4-92e6-f4bf42802cb5',
   'content-type': 'text/xml',
   'content-length': '589'},
  'RetryAttempts': 0}}

In [66]:
print(gateway_lambda_policy_arn)
print(lambda_function["FunctionArn"])

arn:aws:iam::938176218293:policy/AgentCoreGatewayLambdaInvoke-us-east-1
arn:aws:lambda:us-east-1:938176218293:function:hr-assistant-function-def-us-east-1-938176218293


#### Add your Lambda as a tool

In [67]:
target_response = agentcore_control_client.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name="VacationTool",
    description="HR vacation management tool",
    targetConfiguration={
        "mcp": {
            "lambda": {
                "lambdaArn": lambda_function["FunctionArn"],
                "toolSchema": {
                    "inlinePayload": tool_schema
                }
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "GATEWAY_IAM_ROLE"
        }
    ]
)

print(target_response)

{'ResponseMetadata': {'RequestId': '6aa1f1bd-e5e5-42d2-9175-04e3983256e7', 'HTTPStatusCode': 202, 'HTTPHeaders': {'date': 'Mon, 31 Aug 2026 11:00:25 GMT', 'content-type': 'application/json', 'content-length': '830', 'connection': 'keep-alive', 'x-amzn-requestid': '6aa1f1bd-e5e5-42d2-9175-04e3983256e7', 'x-amz-apigw-id': 'C9u_jE2_oAMENdw=', 'x-amzn-trace-id': 'Root=1-6a955ec9-1a2b20e05e4f6c24356de158'}, 'RetryAttempts': 0}, 'gatewayArn': 'arn:aws:bedrock-agentcore:us-east-1:938176218293:gateway/hr-assistant-gateway-m8pnhqdyaz', 'targetId': 'TTS56MX70C', 'createdAt': datetime.datetime(2026, 8, 31, 11, 0, 25, 752684, tzinfo=tzutc()), 'updatedAt': datetime.datetime(2026, 8, 31, 11, 0, 25, 752694, tzinfo=tzutc()), 'status': 'CREATING', 'name': 'VacationTool', 'description': 'HR vacation management tool', 'targetConfiguration': {'mcp': {'lambda': {'lambdaArn': 'arn:aws:lambda:us-east-1:938176218293:function:hr-assistant-function-def-us-east-1-938176218293', 'toolSchema': {'inlinePayload': [{

In [22]:
target_id = target_response["targetId"]

status_response = agentcore_control_client.get_gateway_target(
    gatewayIdentifier=gateway_id,
    targetId=target_id
)

status_response

NameError: name 'target_response' is not defined

In [70]:
import requests
import json

In [11]:
gateway_url

NameError: name 'gateway_url' is not defined

In [73]:
headers = {
    "Accept": "application/json, text/event-stream",
    "Content-Type": "application/json",
    "MCP-Protocol-Version": "2025-03-26"
}

payload = {
    "jsonrpc": "2.0",
    "id": "list-tools-request",
    "method": "tools/list"
}

response = requests.post(
    gateway_url,
    headers=headers,
    json=payload
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2))

Status: 200
{
  "jsonrpc": "2.0",
  "id": "list-tools-request",
  "result": {
    "tools": [
      {
        "inputSchema": {
          "type": "object",
          "properties": {
            "employee_id": {
              "description": "The employee ID.",
              "type": "integer"
            }
          },
          "required": [
            "employee_id"
          ]
        },
        "name": "VacationTool___get_available_vacations_days",
        "description": "Gets the number of available vacation days for an employee."
      }
    ]
  }
}


#### Call the tool through MCP

In [74]:
payload = {
    "jsonrpc": "2.0",
    "id": "call-vacation-tool",
    "method": "tools/call",
    "params": {
        "name": "VacationTool___get_available_vacations_days",
        "arguments": {
            "employee_id": 1
        }
    }
}

response = requests.post(
    gateway_url,
    headers=headers,
    json=payload
)

print("Status:", response.status_code)
print(json.dumps(response.json(), indent=2))

Status: 200
{
  "jsonrpc": "2.0",
  "id": "call-vacation-tool",
  "result": {
    "isError": false,
    "content": [
      {
        "type": "text",
        "text": "{\"statusCode\":200,\"body\":\"{\\\"employee_id\\\": 1, \\\"available_vacation_days\\\": 17}\"}"
      }
    ]
  }
}


### Create a simple Strands agent

In [2]:
# %pip install --force-reinstall pywin32

In [24]:
import boto3

session = boto3.session.Session()
region = session.region_name

sts_client = boto3.client("sts", region_name=region)
iam_client = boto3.client("iam", region_name=region)
lambda_client = boto3.client("lambda", region_name=region)

agentcore_control_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region
)

account_id = sts_client.get_caller_identity()["Account"]

print("Region:", region)
print("Account:", account_id)

Region: us-east-1
Account: 938176218293


In [25]:
gateway_id = "hr-assistant-gateway-m8pnhqdyaz"

gateway_arn = (
    f"arn:aws:bedrock-agentcore:{region}:{account_id}:"
    f"gateway/{gateway_id}"
)

gateway_url = (
    f"https://{gateway_id}.gateway.bedrock-agentcore."
    f"{region}.amazonaws.com/mcp"
)

print("Gateway ID:", gateway_id)
print("Gateway ARN:", gateway_arn)
print("Gateway URL:", gateway_url)

Gateway ID: hr-assistant-gateway-m8pnhqdyaz
Gateway ARN: arn:aws:bedrock-agentcore:us-east-1:938176218293:gateway/hr-assistant-gateway-m8pnhqdyaz
Gateway URL: https://hr-assistant-gateway-m8pnhqdyaz.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp


In [26]:
lambda_function_name = (
    f"hr-assistant-function-def-{region}-{account_id}"
)

print(lambda_function_name)

hr-assistant-function-def-us-east-1-938176218293


In [27]:
targets_response = agentcore_control_client.list_gateway_targets(
    gatewayIdentifier=gateway_id
)

targets_response

{'ResponseMetadata': {'RequestId': 'a0421585-6241-488a-8e59-d877665b101f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 31 Aug 2026 12:40:26 GMT',
   'content-type': 'application/json',
   'content-length': '232',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'a0421585-6241-488a-8e59-d877665b101f',
   'x-amz-apigw-id': 'C99pJHqVIAMEAbQ=',
   'x-amzn-trace-id': 'Root=1-6a95763a-3ff043a30733bfda3bc2bffc'},
  'RetryAttempts': 0},
 'items': [{'targetId': 'TTS56MX70C',
   'name': 'VacationTool',
   'status': 'READY',
   'description': 'HR vacation management tool',
   'createdAt': datetime.datetime(2026, 8, 31, 11, 0, 25, 752684, tzinfo=tzutc()),
   'updatedAt': datetime.datetime(2026, 8, 31, 11, 0, 26, 797846, tzinfo=tzutc()),
   'targetType': 'LAMBDA'}]}

In [28]:
target_id = next(
    target["targetId"]
    for target in targets_response["items"]
    if target["name"] == "VacationTool"
)

print("Target ID:", target_id)

Target ID: TTS56MX70C


In [29]:
status_response = agentcore_control_client.get_gateway_target(
    gatewayIdentifier=gateway_id,
    targetId=target_id
)

print("Target status:", status_response["status"])

Target status: READY


### Test MCP from Python

In [12]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async def test_gateway():

    async with streamable_http_client(
        gateway_url
    ) as (read_stream, write_stream):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            tools = await session.list_tools()

            print("Available tools:")

            for tool in tools.tools:
                print(f"- {tool.name}: {tool.description}")

In [13]:
await test_gateway()

  + Exception Group Traceback (most recent call last):
  |   File "D:\AWS_GenAI\exercise1\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\dhruv\AppData\Local\Temp\ipykernel_30200\3978518691.py", line 1, in <module>
  |     await test_gateway()
  |   File "C:\Users\dhruv\AppData\Local\Temp\ipykernel_30200\368928040.py", line 6, in test_gateway
  |     async with streamable_http_client(
  |                ^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Users\dhruv\anaconda3\Lib\contextlib.py", line 231, in __aexit__
  |     await self.gen.athrow(value)
  |   File "D:\AWS_GenAI\exercise1\Lib\site-packages\mcp\client\streamable_http.py", line 647, in streamable_http_client
  |     async with anyio.create_task_group() as tg:
  |                ^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "D:\AWS_GenAI\exercise1\Lib\site-packages\anyio\_backends\_asyncio.py", line 815, in __aexit__
  |     raise B

In [10]:
async def call_vacation_tool(employee_id):

    async with streamable_http_client(
        gateway_url
    ) as (read_stream, write_stream):

        async with ClientSession(
            read_stream,
            write_stream
        ) as session:

            await session.initialize()

            result = await session.call_tool(
                "VacationTool___get_available_vacations_days",
                arguments={
                    "employee_id": employee_id
                }
            )

            return result

In [11]:
result = await call_vacation_tool(1)

print(result)

  + Exception Group Traceback (most recent call last):
  |   File "D:\AWS_GenAI\exercise1\Lib\site-packages\IPython\core\interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\dhruv\AppData\Local\Temp\ipykernel_30200\563790192.py", line 1, in <module>
  |     result = await call_vacation_tool(1)
  |              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Users\dhruv\AppData\Local\Temp\ipykernel_30200\3498190792.py", line 3, in call_vacation_tool
  |     async with streamable_http_client(
  |                ^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Users\dhruv\anaconda3\Lib\contextlib.py", line 231, in __aexit__
  |     await self.gen.athrow(value)
  |   File "D:\AWS_GenAI\exercise1\Lib\site-packages\mcp\client\streamable_http.py", line 647, in streamable_http_client
  |     async with anyio.create_task_group() as tg:
  |                ^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "D:\AWS_GenAI\exercise1\Lib\site-packages\a

In [38]:
# %pip uninstall -y mcp
%pip install "mcp>=1.1.3,<2.0.0"

  Using cached mcp-1.29.1-py3-none-any.whl.metadata (9.3 kB)
Using cached mcp-1.29.1-py3-none-any.whl (224 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
import mcp
import importlib.metadata

print("MCP version:", importlib.metadata.version("mcp"))

MCP version: 1.29.1


In [17]:
import strands
import inspect

from strands.tools.mcp import MCPClient

print("Strands version:", getattr(strands, "__version__", "unknown"))
print("MCPClient:", MCPClient)

print(inspect.signature(MCPClient))

Strands version: unknown
MCPClient: <class 'strands.tools.mcp.mcp_client.MCPClient'>
(transport_callable: collections.abc.Callable[[], contextlib.AbstractAsyncContextManager[tuple[anyio.streams.memory.MemoryObjectReceiveStream[mcp.shared.message.SessionMessage | Exception], anyio.streams.memory.MemoryObjectSendStream[mcp.shared.message.SessionMessage]] | tuple[anyio.streams.memory.MemoryObjectReceiveStream[mcp.shared.message.SessionMessage | Exception], anyio.streams.memory.MemoryObjectSendStream[mcp.shared.message.SessionMessage], collections.abc.Callable[[], str | None]]]] | None = None, *, url: str | None = None, headers: dict[str, str] | None = None, auth: strands.tools.mcp.mcp_types.MCPClientCredentials | None = None, auth_provider: httpx.Auth | None = None, startup_timeout: int = 30, tool_filters: strands.tools.mcp.mcp_client.ToolFilters | None = None, prefix: str | None = None, application_name: str | None = None, application_version: str | None = None, continue_on_error: bool =

In [18]:
from strands import Agent
from strands.tools.mcp import MCPClient

mcp_client = MCPClient(
    url=gateway_url,
    headers={
        "MCP-Protocol-Version": "2025-03-26"
    }
)

print("MCP client created successfully")

MCP client created successfully


In [19]:
with mcp_client:
    tools = mcp_client.list_tools_sync()

    print("Tools discovered by Strands:")

    for tool in tools:
        print("-", tool.tool_name)

Tools discovered by Strands:
- VacationTool___get_available_vacations_days


In [21]:
inference_profile = "us.amazon.nova-micro-v1:0" 
foundation_model = inference_profile[3:]
foundation_model

'amazon.nova-micro-v1:0'

In [22]:
with mcp_client:

    tools = mcp_client.list_tools_sync()

    agent = Agent(
        model=inference_profile,
        tools=tools,
        system_prompt="""
You are an HR assistant.

You help employees understand HR policies
and manage vacation time.

When an employee asks about their available
vacation days, always use the vacation tool.

Never guess or invent vacation information.
The employee ID must be provided to the tool.

After receiving the tool result, give the user
a concise and clear answer.
"""
    )

    response = agent(
        "How much vacation does the employee with employee_id 1 have available?"
    )

    print(response)

<thinking> To determine the available vacation days for the employee with employee ID 1, I will use the provided tool VacationTool___get_available_vacations_days. I will pass the employee_id as an argument to this tool. </thinking>

Tool #1: VacationTool___get_available_vacations_days
<thinking> The tool has returned the available vacation days for the employee with ID 1. The result indicates that this employee has 17 available vacation days. I will now provide this information directly to the user. </thinking>
You have 17 vacation days available.<thinking> The tool has returned the available vacation days for the employee with ID 1. The result indicates that this employee has 17 available vacation days. I will now provide this information directly to the user. </thinking>
You have 17 vacation days available.



In [31]:
import boto3

iam_client = boto3.client("iam", region_name=region)
lambda_client = boto3.client("lambda", region_name=region)
agentcore_control_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region
)

print("AWS clients ready")

AWS clients ready


In [32]:
gateway_id = "hr-assistant-gateway-m8pnhqdyaz"

print("Gateway ID:", gateway_id)

Gateway ID: hr-assistant-gateway-m8pnhqdyaz


In [33]:
lambda_name = "hr-assistant-function-def-us-east-1-938176218293"

lambda_response = lambda_client.get_function(
    FunctionName=lambda_name
)

lambda_arn = lambda_response["Configuration"]["FunctionArn"]

print("Lambda ARN:", lambda_arn)

Lambda ARN: arn:aws:lambda:us-east-1:938176218293:function:hr-assistant-function-def-us-east-1-938176218293


In [37]:
targets_response = agentcore_control_client.list_gateway_targets(
    gatewayIdentifier=gateway_id
)

for target in targets_response.get("targets", []):
    print(
        "Target ID:", target.get("targetId"),
        "| Name:", target.get("name"),
        "| Status:", target.get("status")
    )

In [38]:
target_id = next(
    target["targetId"]
    for target in targets_response["items"]
    if target["name"] == "VacationTool"
)

print("Target ID:", target_id)

Target ID: TTS56MX70C


In [40]:
roles_response = iam_client.list_roles()

for role in roles_response["Roles"]:
    if "gateway" in role["RoleName"].lower() or "agentcore" in role["RoleName"].lower():
        print(role["RoleName"], "|", role["Arn"])

AgentCoreGatewayRole-us-east-1 | arn:aws:iam::938176218293:role/AgentCoreGatewayRole-us-east-1


In [41]:
delete_target_response = agentcore_control_client.delete_gateway_target(
    gatewayIdentifier=gateway_id,
    targetId=target_id
)

print(delete_target_response)

{'ResponseMetadata': {'RequestId': '93144088-1e8a-4803-b50f-d650dcde1b1b', 'HTTPStatusCode': 202, 'HTTPHeaders': {'date': 'Mon, 31 Aug 2026 12:49:32 GMT', 'content-type': 'application/json', 'content-length': '149', 'connection': 'keep-alive', 'x-amzn-requestid': '93144088-1e8a-4803-b50f-d650dcde1b1b', 'x-amz-apigw-id': 'C9--cFqoIAMErWA=', 'x-amzn-trace-id': 'Root=1-6a95785b-08bbdbed2999417e1d43d571'}, 'RetryAttempts': 0}, 'gatewayArn': 'arn:aws:bedrock-agentcore:us-east-1:938176218293:gateway/hr-assistant-gateway-m8pnhqdyaz', 'targetId': 'TTS56MX70C', 'status': 'DELETING'}


In [42]:
targets_response = agentcore_control_client.list_gateway_targets(
    gatewayIdentifier=gateway_id
)

print(targets_response.get("targets", []))

[]


In [43]:
delete_gateway_response = agentcore_control_client.delete_gateway(
    gatewayIdentifier=gateway_id
)

print(delete_gateway_response)

{'ResponseMetadata': {'RequestId': '39a79416-5008-4492-8278-d378791fdbfc', 'HTTPStatusCode': 202, 'HTTPHeaders': {'date': 'Mon, 31 Aug 2026 12:50:05 GMT', 'content-type': 'application/json', 'content-length': '67', 'connection': 'keep-alive', 'x-amzn-requestid': '39a79416-5008-4492-8278-d378791fdbfc', 'x-amz-apigw-id': 'C9_DrHO4IAMEHGQ=', 'x-amzn-trace-id': 'Root=1-6a95787d-3c08bfd60078fa2800339f31'}, 'RetryAttempts': 0}, 'gatewayId': 'hr-assistant-gateway-m8pnhqdyaz', 'status': 'DELETING'}


In [44]:
try:
    agentcore_control_client.get_gateway(
        gatewayIdentifier=gateway_id
    )
except Exception as e:
    print(type(e).__name__)
    print(e)

ResourceNotFoundException
An error occurred (ResourceNotFoundException) when calling the GetGateway operation: Failed to retrieve gateway because it doesn't exist. Retry the request with a different resource identifier.


In [45]:
lambda_name = "hr-assistant-function-def-us-east-1-938176218293"

delete_lambda_response = lambda_client.delete_function(
    FunctionName=lambda_name
)

print(delete_lambda_response)

{'ResponseMetadata': {'RequestId': '469b1855-0cf7-451c-bd09-f74f341e5207', 'HTTPStatusCode': 204, 'HTTPHeaders': {'date': 'Mon, 31 Aug 2026 12:53:43 GMT', 'content-type': 'application/json', 'connection': 'keep-alive', 'x-amzn-requestid': '469b1855-0cf7-451c-bd09-f74f341e5207'}, 'RetryAttempts': 0}, 'StatusCode': 204}


In [46]:
try:
    lambda_client.get_function(
        FunctionName=lambda_name
    )
except Exception as e:
    print(type(e).__name__)
    print(e)

ResourceNotFoundException
An error occurred (ResourceNotFoundException) when calling the GetFunction operation: Function not found: arn:aws:lambda:us-east-1:938176218293:function:hr-assistant-function-def-us-east-1-938176218293


In [47]:
roles_response = iam_client.list_roles()

for role in roles_response["Roles"]:
    if (
        "AgentCore" in role["RoleName"]
        or "hr-assistant" in role["RoleName"].lower()
        or "lambda" in role["RoleName"].lower()
    ):
        print(
            "Role:",
            role["RoleName"],
            "| ARN:",
            role["Arn"]
        )

Role: AgentCoreGatewayRole-us-east-1 | ARN: arn:aws:iam::938176218293:role/AgentCoreGatewayRole-us-east-1
Role: AmazonBedrockExecutionRoleForAgents_hr-assistant-function-def | ARN: arn:aws:iam::938176218293:role/AmazonBedrockExecutionRoleForAgents_hr-assistant-function-def
Role: hr-assistant-function-def-lambda-role-us-east-1-938176218293 | ARN: arn:aws:iam::938176218293:role/hr-assistant-function-def-lambda-role-us-east-1-938176218293


In [48]:
role_names = [
    "AgentCoreGatewayRole-us-east-1",
    "AmazonBedrockExecutionRoleForAgents_hr-assistant-function-def",
    "hr-assistant-function-def-lambda-role-us-east-1-938176218293"
]

for role_name in role_names:
    print("\n" + "=" * 70)
    print("ROLE:", role_name)

    attached = iam_client.list_attached_role_policies(
        RoleName=role_name
    )

    print("\nAttached policies:")
    for policy in attached["AttachedPolicies"]:
        print(
            "-",
            policy["PolicyName"],
            "|",
            policy["PolicyArn"]
        )

    inline = iam_client.list_role_policies(
        RoleName=role_name
    )

    print("\nInline policies:")
    for policy_name in inline["PolicyNames"]:
        print("-", policy_name)


ROLE: AgentCoreGatewayRole-us-east-1

Attached policies:
- AgentCoreGatewayLambdaInvoke-us-east-1 | arn:aws:iam::938176218293:policy/AgentCoreGatewayLambdaInvoke-us-east-1

Inline policies:

ROLE: AmazonBedrockExecutionRoleForAgents_hr-assistant-function-def

Attached policies:
- hr-assistant-function-def-ba-us-east-1-938176218293 | arn:aws:iam::938176218293:policy/hr-assistant-function-def-ba-us-east-1-938176218293

Inline policies:

ROLE: hr-assistant-function-def-lambda-role-us-east-1-938176218293

Attached policies:
- AWSLambdaBasicExecutionRole | arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole

Inline policies:


In [49]:
gateway_policy_arn = (
    "arn:aws:iam::938176218293:"
    "policy/AgentCoreGatewayLambdaInvoke-us-east-1"
)

iam_client.detach_role_policy(
    RoleName="AgentCoreGatewayRole-us-east-1",
    PolicyArn=gateway_policy_arn
)

print("Gateway policy detached")

Gateway policy detached


In [50]:
iam_client.delete_policy(
    PolicyArn=gateway_policy_arn
)

print("Gateway policy deleted")

Gateway policy deleted


In [51]:
iam_client.delete_role(
    RoleName="AgentCoreGatewayRole-us-east-1"
)

print("Gateway IAM role deleted")

Gateway IAM role deleted


In [52]:
lambda_role_name = (
    "hr-assistant-function-def-lambda-role-us-east-1-938176218293"
)

iam_client.detach_role_policy(
    RoleName=lambda_role_name,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"
)

print("AWSLambdaBasicExecutionRole detached")

AWSLambdaBasicExecutionRole detached


In [53]:
iam_client.delete_role(
    RoleName=lambda_role_name
)

print("Lambda IAM role deleted")

Lambda IAM role deleted


In [54]:
bedrock_policy_arn = (
    "arn:aws:iam::938176218293:"
    "policy/hr-assistant-function-def-ba-us-east-1-938176218293"
)

iam_client.detach_role_policy(
    RoleName="AmazonBedrockExecutionRoleForAgents_hr-assistant-function-def",
    PolicyArn=bedrock_policy_arn
)

print("Bedrock Agent policy detached")

Bedrock Agent policy detached


In [55]:
iam_client.delete_policy(
    PolicyArn=bedrock_policy_arn
)

print("Bedrock Agent policy deleted")

Bedrock Agent policy deleted


In [56]:
iam_client.delete_role(
    RoleName="AmazonBedrockExecutionRoleForAgents_hr-assistant-function-def"
)

print("Bedrock Agent execution role deleted")

Bedrock Agent execution role deleted


In [57]:
role_names = [
    "AgentCoreGatewayRole-us-east-1",
    "AmazonBedrockExecutionRoleForAgents_hr-assistant-function-def",
    "hr-assistant-function-def-lambda-role-us-east-1-938176218293"
]

for role_name in role_names:
    try:
        iam_client.get_role(RoleName=role_name)
        print("STILL EXISTS:", role_name)
    except iam_client.exceptions.NoSuchEntityException:
        print("DELETED:", role_name)

DELETED: AgentCoreGatewayRole-us-east-1
DELETED: AmazonBedrockExecutionRoleForAgents_hr-assistant-function-def
DELETED: hr-assistant-function-def-lambda-role-us-east-1-938176218293
